### IDX Exchange Team **ds55**
### Beini Lan
# 02_preprocessing - Week 3 Data Preprocessing

This Week 3 work carries forward its thirteen-month dataset window (`2025-05` through `2026-05`), Residential + SingleFamilyResidence filter, non-leaky feature candidate list, and exclusion of the leaky features `ListPrice` / `OriginalListPrice`.

**Week 3 requirements covered**
- Handle missing values by dropping unusable fields, imputing numeric features from the training set, and adding missingness flags.
- Convert categorical fields to numeric encodings (0: false/no/does not apply, 1: true/yes/applies).
- Normalize numerical features with training-set z-scores; skewed fields use `log1p` first.
- Use the most recent month as the test set and experiment with prior 12-month training-window lengths.
- Save a cleaned CSV (aggregated the 13-month range of datasets) for W4 training.

## Setup

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 160)
pd.set_option("display.float_format", "{:,.4f}".format)


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "raw data").exists() and (candidate / "W2 Data Exploration" / "01_exploration_v2.ipynb").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root with raw data and W2 Data Exploration/01_exploration_v2.ipynb")


PROJECT_ROOT = find_project_root()
OUTPUT_DIR = PROJECT_ROOT / "W3 Data Preprocessing"
CLEANED_CSV_DIR = OUTPUT_DIR / "Cleaned SFR CRMLSSold CSVs"
WEEK2_NOTEBOOK = PROJECT_ROOT / "W2 Data Exploration" / "01_exploration_v2.ipynb"
print(f"Project root: {PROJECT_ROOT}")
print(f"Referencing Week 2 notebook: {WEEK2_NOTEBOOK}")


## Feature choices from the newer Week 2 notebook

In [ ]:
EXPECTED_MONTHS = [
    "202505", "202506", "202507", "202508", "202509",
    "202510", "202511", "202512",
    "202601", "202602", "202603", "202604", "202605",
]

REQUIRED_COLUMNS = [
    "ListingKey", "ListingId", "CloseDate", "ClosePrice",
    "PropertyType", "PropertySubType", "MlsStatus",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "DaysOnMarket", "City", "CountyOrParish", "PostalCode",
    "Latitude", "Longitude", "YearBuilt", "FireplacesTotal",
    "GarageSpaces", "ParkingTotal", "AssociationFee",
    "NewConstructionYN", "PoolPrivateYN", "ViewYN", "WaterfrontYN",
]

NUMERIC_FEATURES = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "DaysOnMarket", "YearBuilt",
    "Latitude", "Longitude", "GarageSpaces", "ParkingTotal",
    "AssociationFee",
]
SKEWED_NUMERIC_FEATURES = {"LotSizeSquareFeet", "DaysOnMarket", "AssociationFee"}
BOOLEAN_FEATURES = ["NewConstructionYN", "PoolPrivateYN", "ViewYN", "WaterfrontYN"]
CATEGORICAL_FEATURES = ["CountyOrParish", "PostalCode"]
DROPPED_FEATURES = {"FireplacesTotal": "100% missing in the filtered v2 Week 2 dataset"}
LEAKAGE_COLUMNS = ["ListPrice", "OriginalListPrice"]
CHOSEN_TRAINING_WINDOW_MONTHS = 12

assert not set(LEAKAGE_COLUMNS).intersection(NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES)

## Load and filter CRMLS sold data

In [ ]:
def find_crmls_files(expected_months=EXPECTED_MONTHS):
    raw_dir = PROJECT_ROOT / "raw data"
    month_to_file = {}
    for file_path in sorted(raw_dir.glob("CRMLSSold*.csv")):
        month_match = re.search(r"(\d{6})", file_path.stem)
        if month_match and month_match.group(1) in expected_months:
            month_to_file[month_match.group(1)] = file_path
    missing = [month for month in expected_months if month not in month_to_file]
    if missing:
        raise FileNotFoundError(f"Missing expected CRMLS sold files for: {missing}")
    return [month_to_file[month] for month in expected_months]


def month_from_path(path):
    return re.search(r"(\d{6})", path.stem).group(1)


def read_month(path):
    header = pd.read_csv(path, nrows=0).columns
    missing = sorted(set(REQUIRED_COLUMNS).difference(header))
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {missing}")
    frame = pd.read_csv(
        path,
        usecols=REQUIRED_COLUMNS,
        dtype={
            "ListingKey": "string",
            "ListingId": "string",
            "PostalCode": "string",
            "City": "string",
            "CountyOrParish": "string",
            "PropertyType": "string",
            "PropertySubType": "string",
            "MlsStatus": "string",
        },
        low_memory=False,
    )
    frame["SourceFile"] = path.name
    frame["SourceMonth"] = month_from_path(path)
    return frame


DATA_FILES = find_crmls_files()
raw = pd.concat([read_month(path) for path in DATA_FILES], ignore_index=True)

numeric_columns = [
    "ClosePrice", "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "DaysOnMarket",
    "Latitude", "Longitude", "YearBuilt", "FireplacesTotal",
    "GarageSpaces", "ParkingTotal", "AssociationFee",
]
for column in numeric_columns:
    raw[column] = pd.to_numeric(raw[column], errors="coerce")

raw["CloseDate"] = pd.to_datetime(raw["CloseDate"], errors="coerce")
raw["CloseMonth"] = raw["CloseDate"].dt.to_period("M").astype("string")
raw["PropertyFilterMatch"] = raw["PropertyType"].eq("Residential") & raw["PropertySubType"].eq("SingleFamilyResidence")

filtered = raw.loc[
    raw["PropertyFilterMatch"] & raw["ClosePrice"].notna() & raw["ClosePrice"].gt(0)
].copy()

load_summary = pd.DataFrame(
    {
        "metric": [
            "monthly_files_loaded",
            "first_month",
            "last_month",
            "raw_rows",
            "filtered_sfr_rows_with_valid_close_price",
        ],
        "value": [
            len(DATA_FILES),
            min(EXPECTED_MONTHS),
            max(EXPECTED_MONTHS),
            len(raw),
            len(filtered),
        ],
    }
)
display(load_summary)
display(filtered.groupby("SourceMonth").size().rename("filtered_rows").reset_index())

## Missing-value and invalid-value rules

- Rows are restricted to `PropertyType == "Residential"` and `PropertySubType == "SingleFamilyResidence"`.
- Rows with missing or non-positive `ClosePrice` are excluded because `ClosePrice` is the target.
- `FireplacesTotal` is dropped because it is fully missing in the filtered Week 2 v2 data (filtered raw).
- Invalid feature values are converted to missing before imputation: zero living area, zero beds/baths, zero `LotSizeSquareFeet`, negative days on market, out-of-California coordinates, negative parking values, and implausibly large garage/parking counts.
- Numeric missing values are median-imputed using training rows only.
- Missingness flags are retained so future models can learn whether missingness carries effect.

## Train/test split and training-window experiment

In [1]:
def sanitize_feature_values(frame):
    cleaned = frame.copy()
    cleaned.loc[cleaned["LivingArea"].le(0), "LivingArea"] = np.nan
    cleaned.loc[cleaned["BedroomsTotal"].le(0), "BedroomsTotal"] = np.nan
    cleaned.loc[cleaned["BathroomsTotalInteger"].le(0), "BathroomsTotalInteger"] = np.nan
    cleaned.loc[cleaned["LotSizeSquareFeet"].le(0), "LotSizeSquareFeet"] = np.nan
    cleaned.loc[cleaned["DaysOnMarket"].lt(0), "DaysOnMarket"] = np.nan
    cleaned.loc[~cleaned["Latitude"].between(32, 42.5), "Latitude"] = np.nan
    cleaned.loc[~cleaned["Longitude"].between(-125, -113), "Longitude"] = np.nan
    cleaned.loc[cleaned["GarageSpaces"].lt(0) | cleaned["GarageSpaces"].gt(20), "GarageSpaces"] = np.nan
    cleaned.loc[cleaned["ParkingTotal"].lt(0) | cleaned["ParkingTotal"].gt(30), "ParkingTotal"] = np.nan
    return cleaned


cleaned_for_diagnostics = sanitize_feature_values(filtered)

available_months = sorted(cleaned_for_diagnostics["SourceMonth"].dropna().unique().tolist())
TEST_MONTH = available_months[-1]
CANDIDATE_WINDOWS = [window for window in [3, 6, 9, 12] if window <= len(available_months) - 1]

diagnostics = []
test = cleaned_for_diagnostics.loc[cleaned_for_diagnostics["SourceMonth"].eq(TEST_MONTH)]
for window in CANDIDATE_WINDOWS:
    train_months = available_months[-1 - window : -1]
    train = cleaned_for_diagnostics.loc[cleaned_for_diagnostics["SourceMonth"].isin(train_months)]
    iqr_train = train["ClosePrice"].quantile(0.75) - train["ClosePrice"].quantile(0.25)
    iqr_test = test["ClosePrice"].quantile(0.75) - test["ClosePrice"].quantile(0.25)
    diagnostics.append(
        {
            "training_window_months": window,
            "train_months": f"{train_months[0]}-{train_months[-1]}",
            "test_month": TEST_MONTH,
            "train_rows": len(train),
            "test_rows": len(test),
            "median_price_pct_diff_vs_test": (
                (train["ClosePrice"].median() - test["ClosePrice"].median()) / test["ClosePrice"].median() * 100
            ),
            "iqr_pct_diff_vs_test": (iqr_train - iqr_test) / iqr_test * 100,
            "county_coverage_pct": test["CountyOrParish"].isin(set(train["CountyOrParish"].dropna())).mean() * 100,
            "postal_code_coverage_pct": test["PostalCode"].isin(set(train["PostalCode"].dropna())).mean() * 100,
            "avg_feature_missing_pct": train[
                NUMERIC_FEATURES + BOOLEAN_FEATURES + CATEGORICAL_FEATURES
            ].isna().mean().mean() * 100,
        }
    )

window_diagnostics = pd.DataFrame(diagnostics)
display(window_diagnostics)

TRAIN_MONTHS = available_months[-1 - CHOSEN_TRAINING_WINDOW_MONTHS : -1]
print(
    f"Chosen X={CHOSEN_TRAINING_WINDOW_MONTHS}: train months {TRAIN_MONTHS[0]}-{TRAIN_MONTHS[-1]}, "
    f"test month {TEST_MONTH}."
)

NameError: name 'filtered' is not defined

The split experiment keeps `2026-05` as the required holdout month and compares 3, 6, 9, and 12 preceding-month training windows.

`X = 12` is selected for the cleaned CSV because it preserves a full seasonal cycle, has the largest training sample, and gives the strongest county/ZIP coverage for the May 2026 test month while keeping median price drift within a few percentage points of the shorter windows.

## Preprocess features

In [ ]:
def bool_to_numeric(series):
    normalized = series.astype("string").str.strip().str.lower()
    return normalized.map(
        {
            "true": 1, "false": 0,
            "yes": 1, "no": 0,
            "y": 1, "n": 0,
            "1": 1, "0": 0,
        }
    )


def slugify(value):
    slug = re.sub(r"[^0-9a-zA-Z]+", "_", str(value).strip().lower()).strip("_")
    return slug or "missing"


modeling = cleaned_for_diagnostics.loc[cleaned_for_diagnostics["SourceMonth"].isin([*TRAIN_MONTHS, TEST_MONTH])].copy()
modeling["Split"] = np.where(modeling["SourceMonth"].eq(TEST_MONTH), "test", "train")
train_mask = modeling["Split"].eq("train")

cleaned = modeling[["ListingKey", "ListingId", "CloseDate", "SourceMonth", "Split", "ClosePrice"]].copy()
feature_manifest = []

for feature in NUMERIC_FEATURES:
    missing_col = f"{feature}_missing"
    cleaned[missing_col] = modeling[feature].isna().astype("int8")
    median_value = modeling.loc[train_mask, feature].median()
    if pd.isna(median_value):
        median_value = 0
    imputed = modeling[feature].fillna(median_value)

    if feature in SKEWED_NUMERIC_FEATURES:
        transformed = np.log1p(imputed.clip(lower=0))
        feature_col = f"{feature}_log_z"
        transform = "median impute from train; log1p; z-score from train"
    else:
        transformed = imputed
        feature_col = f"{feature}_z"
        transform = "median impute from train; z-score from train"

    mean_value = transformed.loc[train_mask].mean()
    std_value = transformed.loc[train_mask].std(ddof=0)
    if pd.isna(std_value) or std_value == 0:
        std_value = 1
    cleaned[feature_col] = ((transformed - mean_value) / std_value).round(6)
    feature_manifest.extend(
        [
            {"feature_column": feature_col, "source_column": feature, "transform": transform},
            {"feature_column": missing_col, "source_column": feature, "transform": "1 if original value was missing/invalid"},
        ]
    )

for feature in BOOLEAN_FEATURES:
    numeric_values = bool_to_numeric(modeling[feature])
    missing_col = f"{feature}_missing"
    encoded_col = f"{feature}_flag"
    cleaned[missing_col] = numeric_values.isna().astype("int8")
    cleaned[encoded_col] = numeric_values.fillna(0).astype("int8")
    feature_manifest.extend(
        [
            {"feature_column": encoded_col, "source_column": feature, "transform": "True/Yes=1; False/No/missing=0"},
            {"feature_column": missing_col, "source_column": feature, "transform": "1 if original value was missing"},
        ]
    )

county_values = modeling["CountyOrParish"].astype("string").str.strip()
county_values = county_values.mask(county_values.eq(""))
cleaned["CountyOrParish_missing"] = county_values.isna().astype("int8")
feature_manifest.append(
    {"feature_column": "CountyOrParish_missing", "source_column": "CountyOrParish", "transform": "1 if original value was missing"}
)

for county in sorted(county_values.loc[train_mask].dropna().unique().tolist()):
    county_col = f"CountyOrParish__{slugify(county)}"
    cleaned[county_col] = county_values.eq(county).astype("int8")
    feature_manifest.append(
        {
            "feature_column": county_col,
            "source_column": "CountyOrParish",
            "transform": f"one-hot county category from training data: {county}",
        }
    )

postal_values = modeling["PostalCode"].astype("string").str.strip()
postal_values = postal_values.mask(postal_values.eq(""))
cleaned["PostalCode_missing"] = postal_values.isna().astype("int8")
postal_train_frequency = postal_values.loc[train_mask].fillna("Missing").value_counts(normalize=True)
cleaned["PostalCode_train_frequency"] = postal_values.fillna("Missing").map(postal_train_frequency).fillna(0).round(8)
feature_manifest.extend(
    [
        {
            "feature_column": "PostalCode_train_frequency",
            "source_column": "PostalCode",
            "transform": "frequency encoding fit on training rows only; unseen test ZIPs=0",
        },
        {"feature_column": "PostalCode_missing", "source_column": "PostalCode", "transform": "1 if original value was missing"},
    ]
)

cleaned = cleaned.sort_values(["SourceMonth", "ListingKey"], kind="mergesort").reset_index(drop=True)
feature_manifest = pd.DataFrame(feature_manifest).drop_duplicates("feature_column").sort_values("feature_column")

display(cleaned.head())
print(f"Cleaned shape: {cleaned.shape[0]:,} rows x {cleaned.shape[1]:,} columns")
print(f"Feature columns available for modeling: {len(feature_manifest):,}")

## Save Week 3 deliverables

In [ ]:
CLEANED_CSV_DIR.mkdir(exist_ok=True)

cleaned_csv_path = CLEANED_CSV_DIR / "cleaned_crmls_sfr_train_test.csv"

cleaned.to_csv(cleaned_csv_path, index=False)

print(f"Saved cleaned modeling CSV: {cleaned_csv_path}")

## Quality checks

In [ ]:
feature_columns = feature_manifest["feature_column"].tolist()
qc_summary = pd.DataFrame(
    {
        "check": [
            "rows_in_cleaned",
            "train_rows",
            "test_rows",
            "feature_columns",
            "missing_values_in_feature_matrix",
            "non_numeric_feature_columns",
            "target_missing",
        ],
        "value": [
            len(cleaned),
            cleaned["Split"].eq("train").sum(),
            cleaned["Split"].eq("test").sum(),
            len(feature_columns),
            int(cleaned[feature_columns].isna().sum().sum()),
            [
                column
                for column in feature_columns
                if not pd.api.types.is_numeric_dtype(cleaned[column])
            ],
            int(cleaned["ClosePrice"].isna().sum()),
        ],
    }
)
display(qc_summary)

display(
    cleaned.groupby("Split")
    .agg(rows=("ListingKey", "size"), first_month=("SourceMonth", "min"), last_month=("SourceMonth", "max"), median_close_price=("ClosePrice", "median"))
    .reset_index()
)

## Week 3 takeaways

- `Split == "train"` contains the 12 months immediately before the holdout month.
- `Split == "test"` contains the most recent available month, `2026-05`.
- Feature columns are numeric; ID/date/split columns are retained for traceability and should not be used as predictors.
- Encoding and normalization parameters are fit on training rows only to avoid test-set leakage.